# Imports

In [1]:
import numpy as np
from typing import Callable, Optional, Tuple, Any
import scipy.io as sio
import matplotlib.pyplot as plt
from datetime import datetime
import mujoco

from robotblockset.tools import motion_for_load_est, load_est, get_rbs_path
from robotblockset.transformations import rot_z, rot_y, rot_x, map_pose
from robotblockset.mujoco.tools_mjcf import print_body_tree

from robotblockset.mujoco.robots_pymujoco import mujoco_scene, panda

np.set_printoptions(formatter={"float": "{: 0.4f}".format})

# Scene and robot selection 

In [2]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/panda_box_scene.xml"
scene = mujoco_scene(MODEL_PATH, show_camera=None)
r = panda(scene=scene)

[RBS_INFO] [1773470906.579036236] [panda_PyMuJoCo]: Robot connected to MuJoCo


In this model we have placed the F/T sensor at the flange, but rotated as the gripper. Therefore, we have to define correct FT sensor frame

In [3]:
r.SetFTFrame(rot_z(-np.pi / 4))
r.GetFTFrame()

array([[ 0.7071,  0.7071,  0.0000,  0.0000],
       [-0.7071,  0.7071,  0.0000,  0.0000],
       [ 0.0000,  0.0000,  1.0000,  0.0000],
       [ 0.0000,  0.0000,  0.0000,  1.0000]])

In [4]:
print_body_tree(scene.spec.worldbody, scene.spec)

Body Tree for "world"
-Target
-Box (Joints: Box-FREE)
-Load (Joints: Load-FREE)
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_hand
------------panda_left_finger (Joints: panda_finger_joint1-SLIDE)
------------panda_right_finger (Joints: panda_finger_joint2-SLIDE)
------------Panda_tool


Set Load and robot hand masses

In [5]:
r.scene.model.body("Load").mass = 0.4
# r.scene.model.body("panda_hand").mass  = 0.0001
# r.scene.model.body("panda_left_finger").mass =0.0001
# r.scene.model.body("panda_left_finger").mass = 0.0001

Move to initial configuration

In [6]:
r.JMove(r.q_home,2)

0

Without load compensation the gripper mass contributes to the F/T values

In [7]:
r.GetState()
print("Fa : ", r._actual.FT)
print("Ft : ", r.GetFT(task_space="Tool"))
print("Fw : ", r.GetFT(task_space="World"))

Fa :  [-0.1170  0.0230 -8.0722 -0.0009 -0.0857 -0.0033]
Ft :  [-0.1170  0.0230 -8.0722  0.0015 -0.0736 -0.0033]
Fw :  [-0.0427 -0.0196  8.0668  0.0106 -2.4207 -0.0028]


# Random robot tool motion 

Select motion generation function for load estimation.
Execute random motion and capture rotation of the F/T sensor and forces and torques 

In [8]:
Ft, Rt = motion_for_load_est(r)

1 of 50
2 of 50
3 of 50
4 of 50
5 of 50
6 of 50
7 of 50
8 of 50
9 of 50
10 of 50
11 of 50
12 of 50
13 of 50
14 of 50
15 of 50
16 of 50
17 of 50
18 of 50
19 of 50
20 of 50
21 of 50
22 of 50
23 of 50
24 of 50
25 of 50
26 of 50
27 of 50
28 of 50
29 of 50
30 of 50
31 of 50
32 of 50
33 of 50
34 of 50
35 of 50
36 of 50
37 of 50
38 of 50
39 of 50
40 of 50
41 of 50
42 of 50
43 of 50
44 of 50
45 of 50
46 of 50
47 of 50
48 of 50
49 of 50
50 of 50


# Estimate load

In [9]:
mass, COM, offset = load_est(-Ft, Rt)
print("Estimated mass:\n", mass)
print("Estimated center of mass:\n", COM)
print("Sensor offset:\n", offset)

Estimated mass:
 0.7609946355338161
Estimated center of mass:
 [-0.0096  0.0000  0.0312]
Sensor offset:
 [ 0.0000 -0.0000  0.0000  0.0000  0.0000  0.0000]


# Verification

Set estimated load values 

In [10]:
r.Load.mass = mass
r.Load.COM = COM

Move robot

In [11]:
r.JMove(r.q_home)
R0 = r.R_ref

In [12]:
q1=r.q_home.copy()
q1[5] += np.pi / 2
r.JMove(q1)
R0 = r.R_ref

In [13]:
r.CMoveFor(R0 @ r.R.T)
r.CMoveFor(R0 @ r.R.T)

0

# Verify measured forces and torques.

Values without load

In [14]:
r.GetState()
print("Fa : ", r._actual.FT)
print("Ft : ", r.GetFT(task_space="Tool"))
print("Fw : ", r.GetFT(task_space="World"))

Fa :  [ 7.5213 -0.0995  0.0513  0.0032  0.2389  0.0014]
Ft :  [ 0.0565 -0.0985 -0.0381 -0.0070 -0.0008  0.0014]
Fw :  [-0.0387  0.0986  0.0561 -0.0755 -0.0536  0.0355]


Values with Load

> Note that load is 0.1m from TCP 

In [15]:
r.SetEquality("Grasp_load", True)
r.Wait(1)

In [16]:
r.GetState()
print("Fa : ", r._actual.FT)
print("Ft : ", r.GetFT(task_space="Tool"))
print("Fw : ", r.GetFT(task_space="World"))

Fa :  [ 11.3858 -0.0046  0.0649  0.0002 -0.1448  0.0004]
Ft :  [ 3.9206 -0.0032  0.0224 -0.0001 -0.7837  0.0003]
Fw :  [ 0.0001  0.0024  3.9206 -0.0007 -0.9066  0.0011]
